# Deep Hedging a European Call

A reproducible Phase 1 demonstration of dynamic hedging under proportional transaction costs. All reported P&L is discounted, after costs, and includes the initial reporting premium.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from deep_hedge_price.config import load_config
from deep_hedge_price.experiments import load_artifact_manifest
from deep_hedge_price.plotting import *
from deep_hedge_price.policy import architecture_rows
from deep_hedge_price.training import checkpoint_directory, load_policy
project_root = Path.cwd()
config = load_config(project_root / 'configs' / 'quick.yaml')
manifest = load_artifact_manifest(config, project_root)
summary_payload = json.loads(manifest['summary_metrics'].read_text())
sanity = json.loads(manifest['sanity_checks'].read_text())
main_results = pd.read_csv(manifest['main_path_results'])
strategy_summary = pd.read_csv(manifest['strategy_summary'], index_col=0)
sensitivity = pd.read_csv(manifest['sensitivity_summary'])
risk_comparison = pd.read_csv(manifest['risk_objective_summary'])
policy_surface = pd.read_csv(manifest['policy_surface'])
trade_scatter = pd.read_csv(manifest['trade_scatter'])
print(f'Loaded {len(main_results):,} path-strategy rows from config {config.fingerprint()}')

Loaded 80,000 path-strategy rows from config 9db17eb0cb6b


## Executive summary

In [2]:
n = strategy_summary.loc['neural_mse']
nh = strategy_summary.loc['no_hedge']
reduction = 1 - n['std_discounted_pnl_after_costs_including_premium'] / nh['std_discounted_pnl_after_costs_including_premium']
warnings = [name for name, value in sanity.items() if isinstance(value, dict) and not value.get('passed', True)]
display(Markdown(f'''- **Out-of-sample comparison:** {config.training.test_paths:,} common paths.
- **Dispersion:** neural hedging changed P&L standard deviation by {reduction:.1%} relative to no hedge.
- **Tail loss:** neural 99% CVaR loss is {n['cvar_loss_99']:.3f}.
- **Sanity checks:** {'all passed' if not warnings else 'warnings: ' + ', '.join(warnings)}.

Claims below follow the generated metrics; a warning is treated as a limitation, not hidden.'''))

- **Out-of-sample comparison:** 20,000 common paths.
- **Dispersion:** neural hedging changed P&L standard deviation by 82.9% relative to no hedge.
- **Tail loss:** neural 99% CVaR loss is 2.046.
- **Sanity checks:** all passed.

Claims below follow the generated metrics; a warning is treated as a limitation, not hidden.

## Financial setup and sign convention

Under the physical measure, $S_{t+\Delta t}=S_t\exp[(\mu-\sigma^2/2)\Delta t+\sigma\sqrt{\Delta t}Z_t]$. For a short call, discounted loss excluding premium is

$$L_T=\widetilde H-\sum_t\delta_t(\widetilde S_{t+1}-\widetilde S_t)+\sum_t e^{-rt}\lambda S_t|\delta_t-\delta_{t-1}|.$$

Economic P&L is the Black–Scholes premium minus this loss. The initial position is zero and the specified convention has no terminal liquidation charge.

### Differentiable training path

In [3]:
training_diagram(); plt.show()

/tmp/ipykernel_303497/34529783.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  training_diagram(); plt.show()


## Configuration

In [4]:
config_rows = []
for section, values in config.to_dict().items():
    if isinstance(values, dict):
        config_rows.extend({'section': section, 'parameter': k, 'value': v} for k, v in values.items())
display(pd.DataFrame(config_rows).head(40))

,section,parameter,value
0,market,seed,1210
1,market,s0,100.0
2,market,strike,100.0
3,market,maturity_years,0.119048
4,market,n_steps,30
5,market,mu,0.05
6,market,risk_free_rate,0.0
7,market,volatility,0.2
8,market,transaction_cost_bps,5.0
9,market,option_position,-1.0


## Market simulation and payoff

In [5]:
sample_paths_figure(config); plt.show()

/tmp/ipykernel_303497/3124603685.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  sample_paths_figure(config); plt.show()


In [6]:
payoff_figure(config); plt.show()

/tmp/ipykernel_303497/3564135246.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  payoff_figure(config); plt.show()


## Neural-policy architecture

In [7]:
model_config = config.with_risk(objective='mse')
policy, _ = load_policy(model_config, checkpoint_directory(model_config, project_root) / 'best.pt', device='cpu')
display(pd.DataFrame(architecture_rows(policy)))
print(f'Total trainable parameters: {policy.parameter_count:,}')

,index,layer,parameters
0,0,Linear,384
1,1,SiLU,0
2,2,Linear,4160
3,3,SiLU,0
4,4,Linear,4160
5,5,SiLU,0
6,6,Linear,65


Total trainable parameters: 8,769


## End-to-end training history

In [8]:
history = pd.read_csv(checkpoint_directory(model_config, project_root) / 'history.csv')
training_history_figure(history); plt.show()

/tmp/ipykernel_303497/291573762.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  training_history_figure(history); plt.show()


## Main comparison on common test paths

In [9]:
comparison_columns = ['mean_discounted_pnl_after_costs_including_premium', 'std_discounted_pnl_after_costs_including_premium', 'rmse_discounted_hedging_error', 'cvar_loss_99', 'average_turnover_shares', 'average_discounted_transaction_cost']
display(strategy_summary[comparison_columns].round(4))

,mean_discounted_pnl_after_costs_including_premium,std_discounted_pnl_after_costs_including_premium,rmse_discounted_hedging_error,cvar_loss_99,average_turnover_shares,average_discounted_transaction_cost
neural_mse,-0.0741,0.7417,0.7454,2.0457,2.0608,0.1034
black_scholes_delta,-0.1061,0.4418,0.4543,1.6743,2.2328,0.1117
black_scholes_band,-0.0664,0.5349,0.5390,1.6580,1.4283,0.0715
no_hedge,-0.2668,4.3494,4.3575,17.5672,0.0000,0.0000


### P&L distributions

In [10]:
pnl_distribution_figure(main_results); plt.show()

/tmp/ipykernel_303497/600233549.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pnl_distribution_figure(main_results); plt.show()


### Empirical CDF and lower tail

In [11]:
ecdf_figure(main_results); plt.show()

/tmp/ipykernel_303497/3860087178.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ecdf_figure(main_results); plt.show()


### VaR and CVaR

In [12]:
var_cvar_figure(strategy_summary); plt.show()

/tmp/ipykernel_303497/211978039.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  var_cvar_figure(strategy_summary); plt.show()


### Turnover and transaction cost

In [13]:
turnover_cost_figure(strategy_summary); plt.show()

/tmp/ipykernel_303497/3154127266.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  turnover_cost_figure(strategy_summary); plt.show()


## Policy surface
The following slice fixes the previous position at 0.5 shares.

In [14]:
policy_heatmap_figure(policy_surface); plt.show()

/tmp/ipykernel_303497/2521431087.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  policy_heatmap_figure(policy_surface); plt.show()


### Neural minus Black–Scholes delta

In [15]:
policy_heatmap_figure(policy_surface, difference=True); plt.show()

/tmp/ipykernel_303497/3747880089.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  policy_heatmap_figure(policy_surface, difference=True); plt.show()


### Policy slices at several maturities

In [16]:
policy_slices_figure(policy_surface); plt.show()

/tmp/ipykernel_303497/1889053825.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  policy_slices_figure(policy_surface); plt.show()


### Trade size conditional on prior position and BS target

In [17]:
trade_scatter_figure(trade_scatter); plt.show()

/tmp/ipykernel_303497/2145327161.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  trade_scatter_figure(trade_scatter); plt.show()


## Transaction-cost sensitivity

In [18]:
display(sensitivity.round(4)); sensitivity_figure(sensitivity); plt.show()

,transaction_cost_bps,n_paths,mean_discounted_pnl_after_costs_including_premium,std_discounted_pnl_after_costs_including_premium,rmse_discounted_hedging_error,mae_discounted_hedging_error,median_discounted_pnl,minimum_discounted_pnl,maximum_discounted_pnl,pnl_quantile_01,...,terminal_delta_std,terminal_delta_min,terminal_delta_q05,terminal_delta_median,terminal_delta_q95,terminal_delta_max,payoff_net_gain_correlation,policy_bs_mae,policy_bs_rmse,checkpoint
0,0.0,20000,0.0249,0.7305,0.7310,0.5600,-0.0040,-3.3286,5.2921,-1.7252,...,0.3920,-0.2142,-0.0157,0.6282,1.1919,1.2498,0.9881,0.0607,0.0854,/home/kazumasa/projects/deep_hedge_price/artif...
1,1.0,20000,0.0038,0.7286,0.7286,0.5617,-0.0320,-3.2936,5.1763,-1.7300,...,0.3864,-0.2115,-0.0068,0.6265,1.1885,1.2498,0.9879,0.0591,0.0851,/home/kazumasa/projects/deep_hedge_price/artif...
2,5.0,20000,-0.0741,0.7417,0.7454,0.5932,-0.1271,-3.1450,4.7311,-1.7657,...,0.3632,-0.1956,0.0338,0.6198,1.1744,1.2497,0.9865,0.0569,0.0882,/home/kazumasa/projects/deep_hedge_price/artif...
3,10.0,20000,-0.1777,0.7458,0.7666,0.6228,-0.2452,-3.1426,4.5820,-1.8184,...,0.3584,-0.1833,0.0472,0.6079,1.1749,1.2498,0.9860,0.0563,0.0884,/home/kazumasa/projects/deep_hedge_price/artif...
4,25.0,20000,-0.2768,0.9183,0.9591,0.7708,-0.3017,-4.5141,2.5213,-2.4582,...,0.3674,-0.0559,0.0511,0.5495,1.1251,1.2308,0.9775,0.0916,0.1156,/home/kazumasa/projects/deep_hedge_price/artif...


/tmp/ipykernel_303497/4289322152.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  display(sensitivity.round(4)); sensitivity_figure(sensitivity); plt.show()


## Risk-objective comparison

In [19]:
display(risk_comparison.round(4)); risk_objective_figure(risk_comparison); plt.show()

,objective,n_paths,mean_discounted_pnl_after_costs_including_premium,std_discounted_pnl_after_costs_including_premium,rmse_discounted_hedging_error,mae_discounted_hedging_error,median_discounted_pnl,minimum_discounted_pnl,maximum_discounted_pnl,pnl_quantile_01,...,average_turnover_shares,average_meaningful_trades,terminal_delta_mean,terminal_delta_std,terminal_delta_min,terminal_delta_q05,terminal_delta_median,terminal_delta_q95,terminal_delta_max,payoff_net_gain_correlation
0,mse,20000,-0.0741,0.7417,0.7454,0.5932,-0.1271,-3.1450,4.7311,-1.7657,...,2.0608,29.6346,0.6165,0.3632,-0.1956,0.0338,0.6198,1.1744,1.2497,0.9865
1,entropic,20000,-0.0056,1.0436,1.0436,0.8377,0.1793,-3.8839,5.1563,-2.5086,...,1.6605,29.5070,0.6656,0.3605,-0.1056,0.0990,0.6729,1.1927,1.2495,0.9821


/tmp/ipykernel_303497/2588677608.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  display(risk_comparison.round(4)); risk_objective_figure(risk_comparison); plt.show()


## Sanity checks and limitations

In [20]:
check_rows = [{'check': k, 'passed': v.get('passed'), 'details': str({x:y for x,y in v.items() if x != 'passed'})} for k,v in sanity.items() if isinstance(v, dict)]
display(pd.DataFrame(check_rows))

,check,passed,details
0,common_random_numbers,True,{'test_seed': 21210}
1,competent_hedge_reduces_dispersion,True,"{'best_hedged_std': 0.4417760975884921, 'no_he..."
2,near_frictionless_closer_to_bs_than_25bp,True,"{'high_cost_bps': 25.0, 'high_rmse': 0.1155780..."
3,separate_train_validation_test,True,"{'test_seed': 21210, 'training_seed_range': [1..."
4,tail_sample_size,True,"{'expected_99pct_tail_observations': 200, 'tes..."
5,turnover_generally_decreases_with_cost,True,"{'high_turnover': 1.1899478435516357, 'low_tur..."


Constant-volatility GBM omits jumps, stochastic volatility, liquidity state, and model risk. Quick-profile tail estimates are intentionally labeled less stable than the full profile. No claim of superiority is made when its supporting sanity check fails.

## Phase 2 roadmap — Deep Pricing

Next, build a separately validated Black–Scholes/Monte Carlo label pipeline, then price-and-Greeks surrogates, differential learning, arbitrage-aware penalties, calibration tests, and inference-speed benchmarks. The executable specification is in `docs/ROADMAP_DEEP_PRICING.md`; Phase 2 code is deliberately out of scope here.